<a href="https://colab.research.google.com/github/your-org/alexpose/blob/main/experiments/multiple-sclerosis/02_anatomical_mask_and_tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Masking and tokenization

S-JEPA learns by hiding part of the skeleton and predicting the hidden part in feature space. Two design choices drive this notebook: **how we cut the skeleton into tokens**, and **which joints we hide**.

> **What changed, and why.** An earlier version of this project hid the *same* twelve clinical joints on every single step. That turned out to be a real bug: the encoder never saw those joints as context, so their internal position settings received no learning signal, yet the classifier then pooled exactly those joints. We now use **stochastic graph-time masks**: a different connected group of joints is hidden each step, so every joint is sometimes context and sometimes a target. Clinical knowledge still guides us, but gently, by choosing the leg and shoulder joints as targets a bit more often. We also do **not** bias toward the busiest joints (the paper's motion-aware masking), because in MS and PD the telling sign is often *reduced* motion, which a high-motion mask would hide.


In [ ]:
# --- Setup: install dependencies (Colab installs; local usually already has them) ---
import importlib, importlib.util, subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

def _need(mod):
    return importlib.util.find_spec(mod) is None

# Light deps used by every notebook.
_pkgs = []
for mod, pip_name in [('cv2','opencv-python'), ('mediapipe','mediapipe'),
                      ('sklearn','scikit-learn'), ('pandas','pandas'),
                      ('matplotlib','matplotlib'), ('tqdm','tqdm')]:
    if _need(mod):
        _pkgs.append(pip_name)
if _pkgs:
    print('installing:', _pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs])
else:
    print('all light dependencies already present')

In [ ]:
# --- Make `sjepa` and `ambient` importable, locally and in Colab ---
from pathlib import Path
import sys, subprocess

def _find_exp_dir():
    # Local run: this notebook sits in experiments/multiple-sclerosis.
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / 'sjepa' / '__init__.py').exists():
            return p
    return None

EXP_DIR = _find_exp_dir()
if EXP_DIR is None:
    # Colab: clone the repo, then point at the experiment folder.
    REPO = 'https://github.com/your-org/alexpose.git'  # <-- edit to your fork
    if not Path('alexpose').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO])
    EXP_DIR = Path('alexpose') / 'experiments' / 'multiple-sclerosis'

REPO_ROOT = EXP_DIR.parents[1]
for p in (str(EXP_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('experiment dir:', EXP_DIR)
print('repo root     :', REPO_ROOT)

In [ ]:
# --- Paths and profile (reads the root .env if python-dotenv is present) ---
import os
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
except Exception:
    pass

VIDEO_DIR = EXP_DIR / 'video-data'
ARTIFACT_DIR = EXP_DIR / 'artifacts'
KEYPOINTS_DIR = ARTIFACT_DIR / 'keypoints'
IMAGES_DIR = EXP_DIR / 'images'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Pick the model size profile. 'laptop' is the fast default; set SJEPA_PROFILE=gpu
# in your .env for a larger model, or SJEPA_SMOKE=1 for a near-instant test run.
os.environ.setdefault('SJEPA_PROFILE', 'laptop')
print('SJEPA_PROFILE =', os.environ['SJEPA_PROFILE'],
      '| SJEPA_SMOKE =', os.environ.get('SJEPA_SMOKE', '0'))

## Tokenizing a window

A training window is a short movie of stick figures. We group `l = 4` adjacent frames of one joint into a single token, so each token summarizes how that joint moved over a moment. With 32 frames and 33 joints that gives `(32 / 4) x 33 = 264` tokens. Token index is `t * V + v` (time block `t`, joint `v`).


In [ ]:
from IPython.display import SVG, display
display(SVG(filename=str(IMAGES_DIR / 'tokenization.svg')))

In [ ]:
from sjepa.config import get_config, describe
cfg = get_config()  # honours SJEPA_PROFILE
print(describe(cfg))
print('tokens per window N =', cfg.num_tokens,
      f'= {cfg.num_time_tokens} time blocks x {cfg.num_joints} joints')

## The clinical joints (domain context, not a permanent mask)

The file `mapping-data/ms-pd-mapping.md` lists the joints clinicians care about for ms and pd. After removing duplicates and sorting, we get exactly twelve BlazePose landmarks: both shoulders and both complete legs. We keep this table as **domain knowledge** that biases how often a joint is chosen as a target, but every joint can still be both context and target.


In [ ]:
from sjepa.masking_v2 import CLINICAL_JOINTS
from ambient.pose.keypoint_data import MEDIAPIPE_33_NAMES
import pandas as pd

features_for = {
    11: 'shoulder_symmetry_index, trunk_lean_angle',
    12: 'shoulder_symmetry_index, trunk_lean_angle',
    23: 'walking_speed_ms, hip_asymmetry, knee_range, trunk_lean_angle',
    24: 'walking_speed_ms, hip_asymmetry, knee_range, trunk_lean_angle',
    25: 'knee_range, ankle_range', 26: 'knee_range, ankle_range',
    27: 'knee_range, ankle_range, step_width_m', 28: 'knee_range, ankle_range, step_width_m',
    29: 'stride_length_m, double_support_pct, stride_time_cv, ankle_range',
    30: 'stride_length_m, double_support_pct, stride_time_cv, ankle_range',
    31: 'stride_length_m, double_support_pct, stride_time_cv, ankle_range',
    32: 'stride_length_m, double_support_pct, stride_time_cv, ankle_range',
}
table = pd.DataFrame([
    {'BLAZEPOSE_33 index': j, 'Keypoint name': MEDIAPIPE_33_NAMES[j],
     'Features involved': features_for[j]}
    for j in sorted(CLINICAL_JOINTS)
])
table

## Stochastic graph-time masks

Each step we sample a per-example mask: connected groups of joints (a limb or the trunk) over a contiguous span of time. The cell below samples a few masks and shows they differ, that every one keeps visible context, and that over a bank of masks every joint is both visible and targeted often enough (the coverage gates).


In [ ]:
import numpy as np
from sjepa.masking_v2 import sample_mask_batch, mask_bank_stats

rng = np.random.default_rng(0)
batch = sample_mask_batch(6, cfg.num_joints, cfg.num_time_tokens, rng)
print('mask batch shape (B, N):', batch.shape)
print('unique masks in the batch:', len({row.tobytes() for row in batch}), 'of 6')
print('every row has context and target:',
      bool((~batch).any(1).all() and batch.any(1).all()))

stats = mask_bank_stats(cfg.num_joints, cfg.num_time_tokens, n_masks=512, seed=0)
print(f'over 512 masks: min joint-visible {stats.joint_visible_frac.min():.2f} '
      f'(gate >=0.20), min joint-target {stats.joint_target_frac.min():.2f} (gate >=0.10)')
print(f'mean target fraction {stats.mean_target_frac:.2f}')

Here is the difference drawn out: a fixed mask hides the same joints forever (left), while stochastic masks rotate which joints are hidden (right).


In [ ]:
display(SVG(filename=str(IMAGES_DIR / 'defect_mask_starvation.svg')))

## See one mask on a real skeleton

The animation highlights one sampled set of masked joints in red on a real walking sequence. Next time you sample, a different group will be hidden.


In [ ]:
from sjepa.data import load_index
from sjepa.masking_v2 import sample_target_mask
from sjepa.viz import skeleton_animation
from IPython.display import Image

recs = load_index(KEYPOINTS_DIR)
seq = recs[0].load_norm()
tgt = sample_target_mask(cfg.num_joints, cfg.num_time_tokens, np.random.default_rng(1))
masked_joints = sorted({int(i % cfg.num_joints) for i in np.nonzero(tgt)[0]})
gif = skeleton_animation(seq, ARTIFACT_DIR / 'mask_demo.gif',
                         masked_joints=masked_joints, fps=15,
                         title='red = one sampled set of masked joints')
Image(filename=str(gif))